In [1]:
!pip install sentencepiece sacrebleu evaluate datasets
!pip install -U transformers==4.44.2 peft==0.10.0 accelerate==0.33.0 bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 39.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow

In [2]:
from peft import LoraConfig, get_peft_model
import os
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from transformers import TrainerCallback
import time
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"


2025-11-08 05:10:05.137645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762578605.331860      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762578605.382867      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
dataset = load_dataset("csv", data_files="/kaggle/input/sin-eng/sin-eng.csv")

dataset = dataset["train"].train_test_split(test_size=0.2)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-1.3B") 
model = AutoModelForSeq2SeqLM.from_pretrained( "facebook/nllb-200-1.3B", 
        torch_dtype=torch.float32, 
        device_map="cuda")
model.to('cuda')


Generating train split: 0 examples [00:00, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-23): 24 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192, out_features=1024, bias=True)
       

In [4]:
source_lang = "sin_Sinh"
target_lang = "eng_Latn"
tokenizer.src_lang = source_lang
tokenizer.tgt_lang = target_lang

max_length = 128
def preprocess(examples):
    # Get raw sentences
    inputs = examples["sinhala"]
    targets = examples["english"]
    
    # Tokenize source and target
    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        padding="max_length",
        truncation=True
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_length,
            padding="max_length",
            truncation=True
        )
    
    # Put labels in model_inputs
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs
# def preprocess(examples):
#     inputs = examples["sinhala"]
#     targets = examples["english"]
#     model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
#     labels = tokenizer(targets, max_length=max_length, truncation=True)
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
eval_dataset = eval_dataset.map(preprocess, batched=True)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)


Map:   0%|          | 0/3881 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/971 [00:00<?, ? examples/s]

In [5]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_sinhala2english",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=5e-4,
    num_train_epochs=20,
    logging_strategy="steps",
    eval_strategy="steps",
    save_strategy="steps",
    logging_dir="./logs",
    predict_with_generate=True,
    fp16=True,
    gradient_accumulation_steps=2,      
    dataloader_pin_memory=True,
    report_to="none",                   
    load_best_model_at_end=True,
    metric_for_best_model="loss",      
    save_total_limit=3,    
)

# Load BLEU metric
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result["bleu"] = result["score"]
    return result

In [6]:
class RealTimeProgressCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        logs = kwargs.get("logs", {})
        if "loss" in logs:
            current_time = time.strftime("%H:%M:%S", time.localtime())
            print(f"[{current_time}] Step: {state.global_step}, Loss: {logs['loss']:.4f}", flush=True)
        if state.global_step % 3880 == 0:
            trainer.save_model(f"./nllb_sinhala2english_gpu/checkpoint-{state.global_step}")


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
# Add the callback to the trainer
trainer.add_callback(RealTimeProgressCallback)

trainer.args.logging_steps = 1 
trainer.args.eval_steps = 3880
trainer.train()

model.save_pretrained("./nllb_sinhala2english_lora")
tokenizer.save_pretrained("./nllb_sinhala2english_lora")

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss,Validation Loss,Score,Counts,Totals,Precisions,Bp,Sys Len,Ref Len,Bleu
3880,6.589400,7.445986,37.984433,"[13273, 8303, 5599, 3852]","[19268, 18297, 17331, 16379]","[68.88623624662654, 45.37902388369678, 32.3062719981536, 23.51791928689175]",0.967581,19268,19903,37.984433
7760,7.141500,7.436420,41.335522,"[13761, 8909, 6185, 4406]","[19438, 18467, 17502, 16553]","[70.79432040333367, 48.242811501597444, 35.33881842075191, 26.617531565275176]",0.976362,19438,19903,41.335522
11640,6.974100,7.444197,43.134186,"[14005, 9206, 6530, 4728]","[19782, 18811, 17846, 16897]","[70.79668385400869, 48.93945032162033, 36.59083267959207, 27.981298455347105]",0.993902,19782,19903,43.134186
15520,8.580300,7.456051,43.599166,"[14108, 9313, 6608, 4781]","[19729, 18758, 17792, 16842]","[71.5089462212986, 49.648150122614354, 37.14028776978417, 28.387364921030755]",0.991219,19729,19903,43.599166
19400,8.226700,7.465772,44.269234,"[14059, 9388, 6735, 4937]","[19333, 18362, 17396, 16445]","[72.72021931412611, 51.127328177758415, 38.71579673488158, 30.021283064761327]",0.970947,19333,19903,44.269234
23280,6.840500,7.477650,44.630333,"[14108, 9465, 6809, 4986]","[19308, 18337, 17372, 16423]","[73.06815827636213, 51.616949337405245, 39.19525673497582, 30.35986117030993]",0.969654,19308,19903,44.630333
27160,8.509000,7.484047,45.265686,"[14276, 9584, 6907, 5103]","[19723, 18752, 17786, 16836]","[72.38249759164427, 51.10921501706485, 38.83391431462948, 30.310049893086244]",0.990915,19723,19903,45.265686
31040,8.001500,7.491497,44.933121,"[14239, 9548, 6853, 5022]","[19671, 18700, 17735, 16784]","[72.38574551370037, 51.05882352941177, 38.64110515928954, 29.92135367016206]",0.988275,19671,19903,44.933121
34920,7.360000,7.499118,45.186779,"[14201, 9596, 6910, 5077]","[19588, 18617, 17653, 16703]","[72.49846845007147, 51.54428747918569, 39.143488358919164, 30.39573729270191]",0.984047,19588,19903,45.186779
38800,6.744800,7.499716,45.237382,"[14232, 9599, 6909, 5095]","[19695, 18724, 17759, 16809]","[72.26199543031227, 51.26575518051698, 38.904217579818685, 30.31114284014516]",0.989495,19695,19903,45.237382


('./nllb_sinhala2english_lora/tokenizer_config.json',
 './nllb_sinhala2english_lora/special_tokens_map.json',
 './nllb_sinhala2english_lora/sentencepiece.bpe.model',
 './nllb_sinhala2english_lora/added_tokens.json',
 './nllb_sinhala2english_lora/tokenizer.json')

In [10]:
import os
import json
from transformers import AutoTokenizer

save_dir = "./nllb_sinhala2english_lora"

# Create dir if missing
os.makedirs(save_dir, exist_ok=True)

# --- Save model + tokenizer ---
trainer.model.save_pretrained(save_dir)
trainer.tokenizer.save_pretrained(save_dir)

def serialize_training_args(args):
    data = {}
    for k, v in vars(args).items():
        try:
            json.dumps(v)  # test serializability
            data[k] = v
        except TypeError:
            data[k] = str(v)
    return data

training_args_dict = serialize_training_args(trainer.args)

with open(f"{save_dir}/training_args.json", "w") as f:
    json.dump(training_args_dict, f, indent=4)

print(f"✅ Model, tokenizer, and training args saved in: {save_dir}")

✅ Model, tokenizer, and training args saved in: ./nllb_sinhala2english_lora


In [11]:
!zip -r "/kaggle/working/nllb_sinhala2english_lora.zip" "/kaggle/working//nllb_sinhala2english_lora"

updating: kaggle/working//nllb_sinhala2english_lora/ (stored 0%)
updating: kaggle/working//nllb_sinhala2english_lora/adapter_config.json (deflated 51%)
updating: kaggle/working//nllb_sinhala2english_lora/adapter_model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
updating: kaggle/working//nllb_sinhala2english_lora/sentencepiece.bpe.model (deflated 51%)
updating: kaggle/working//nllb_sinhala2english_lora/tokenizer_config.json (deflated 94%)
updating: kaggle/working//nllb_sinhala2english_lora/special_tokens_map.json (deflated 79%)
updating: kaggle/working//nllb_sinhala2english_lora/tokenizer.json (deflated 69%)
updating: kaggle/working//nllb_sinhala2english_lora/training_args.json (deflated 66%)
updating: kaggle/working//nllb_sinhala2english_lora/README.md (deflated 66%)


In [13]:
import json
import os
import shutil

# Paths
WORKING_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(WORKING_DIR, "finetuned")
ZIP_PATH = os.path.join(WORKING_DIR, "nllb_sinhala2english_lora.zip")

# Make sure the output folder exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Move the zip into the output folder
shutil.copy(ZIP_PATH, OUTPUT_DIR)

# Dataset metadata
metadata = {
    "title": "nllb-fine-tuned-1-3B",
    "id": "zeidhnazly/nllb-fine-tuned-1-3B",
    "licenses": [{"name": "CC0-1.0"}]
}

# Save metadata JSON inside the output folder
metadata_path = os.path.join(OUTPUT_DIR, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f)

# Install/upgrade Kaggle API
!pip install kaggle --upgrade

# Set up Kaggle API key
os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp "/kaggle/input/api-key-2/kaggle.json" "/root/.config/kaggle/"
!chmod 600 /root/.config/kaggle/kaggle.json

# Create the Kaggle dataset
!kaggle datasets create -p {OUTPUT_DIR} -u --quiet


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Your public Dataset is being created. Please check progress at https://www.kaggle.com/datasets/zeidhnazly/nllb-fine-tuned-1-3B


In [14]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Define the paths
base_model_name = "facebook/nllb-200-1.3B"
adapter_path = "./nllb_sinhala2english_lora"  # your fine-tuned folder

# Load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name, torch_dtype=torch.float16, device_map="auto")

# Load the fine-tuned LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()  # put model in inference mode


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): M2M100ForConditionalGeneration(
      (model): M2M100Model(
        (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
        (encoder): M2M100Encoder(
          (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
          (embed_positions): M2M100SinusoidalPositionalEmbedding()
          (layers): ModuleList(
            (0-23): 24 x M2M100EncoderLayer(
              (self_attn): M2M100Attention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=16, bias=False)
                  )
                  (lor

In [19]:
source_lang = "sin_Sinh"
target_lang = "eng_Latn"

tokenizer.src_lang = source_lang

def translate(text, max_new_tokens=128):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate translation
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
        max_new_tokens=max_new_tokens,
        num_beams=5,         # beam search for better quality
        length_penalty=1.0,  # can adjust for output length control
    )

    # Decode tokens
    translation = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translation

# Example
text = "අතින් ගෙල සිර කිරීම"
print(translate(text))


Manual strangulation
